In [1]:
import platform
from pathlib import Path

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd

from vascular_superenhancement.utils.path_config import load_path_config, _PROJECT_ROOT

config_name = "local_mac" if platform.system() == "Darwin" else "all_patients"
pc = load_path_config(config_name)

PATIENT_DATA_DIR = pc.working_dir / "patient_data"
DS_FOLDER = "downsampled_full_fov_128x128x64_crop-17.5"

SLICE_INDICES = [0, 8, 16, 24, 32, 40, 48, 63]

MODALITIES = [
    ("3d_cine",          "3d_cine",          "Cine"),
    ("4d_flow_mag",      "4d_flow_mag",      "Mag"),
    ("4d_flow_vx",       "4d_flow_vx",       "Vx"),
    ("4d_flow_vy",       "4d_flow_vy",       "Vy"),
    ("4d_flow_vz",       "4d_flow_vz",       "Vz"),
    ("4d_flow_diff_vx",  "4d_flow_diff_vx",  "Diff Vx"),
    ("4d_flow_diff_vy",  "4d_flow_diff_vy",  "Diff Vy"),
    ("4d_flow_diff_vz",  "4d_flow_diff_vz",  "Diff Vz"),
]

splits_df = pd.read_csv(_PROJECT_ROOT / "splits" / "splits_01-15-26.csv")
patients_df = splits_df[splits_df["split"].isin(["train", "validation", "test"])].copy()
patients_df = patients_df.sort_values(["split", "patient_id"]).reset_index(drop=True)

print(f"Non-skipped patients: {len(patients_df)}")
print(patients_df["split"].value_counts())

Found project root at: /home/ayeluru/vascular-superenhancement-4d-flow
Non-skipped patients: 209
split
train         161
test           28
validation     20
Name: count, dtype: int64


In [2]:
OUTPUT_DIR = Path("slice_order_check_images")
OUTPUT_DIR.mkdir(exist_ok=True)


def load_volume(path: Path) -> np.ndarray:
    return nib.load(str(path)).get_fdata(dtype=np.float32)


for i, (_, row) in enumerate(patients_df.iterrows()):
    pid = row["patient_id"]
    split = row["split"]
    ds_root = PATIENT_DATA_DIR / pid / "nifti" / DS_FOLDER

    if not ds_root.exists():
        print(f"SKIP {pid}: no downsampled dir")
        continue

    n_rows = len(MODALITIES)
    n_cols = len(SLICE_INDICES)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 2.5 * n_rows))
    fig.suptitle(f"{pid}  ({split})", fontsize=14, fontweight="bold", y=1.01)

    for r, (subfolder, prefix, label) in enumerate(MODALITIES):
        nifti_path = ds_root / subfolder / f"{prefix}_{pid}_frame_00.nii.gz"

        if not nifti_path.exists():
            for c in range(n_cols):
                axes[r, c].set_visible(False)
            axes[r, 0].set_visible(True)
            axes[r, 0].text(0.5, 0.5, f"{label}\nMISSING", ha="center", va="center", transform=axes[r, 0].transAxes)
            axes[r, 0].set_axis_off()
            continue

        vol = load_volume(nifti_path)  # (X, Y, Z)
        n_slices = vol.shape[2]

        is_signed = prefix.startswith("4d_flow_v") or "diff" in prefix
        if is_signed:
            vmax = np.percentile(np.abs(vol), 99)
            vmin = -vmax
            cmap = "RdBu_r"
        else:
            vmin = 0
            vmax = np.percentile(vol, 99)
            cmap = "gray"

        for c, s_idx in enumerate(SLICE_INDICES):
            ax = axes[r, c]
            si = min(s_idx, n_slices - 1)
            ax.imshow(vol[:, :, si].T, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax, aspect="equal")
            ax.set_xticks([])
            ax.set_yticks([])
            if r == 0:
                ax.set_title(f"z={si}", fontsize=10)
            if c == 0:
                ax.set_ylabel(label, fontsize=10, rotation=0, labelpad=50, va="center")

    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{split}_{pid}.png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"[{i+1}/{len(patients_df)}] {pid} OK")

print(f"\nDone. Images saved to {OUTPUT_DIR.resolve()}")

[1/209] Balboloop OK
[2/209] Biswifo OK
[3/209] Bomatog OK
[4/209] Boochuto OK
[5/209] Boumorim OK
[6/209] Bovutou OK
[7/209] Cadotueg OK
[8/209] Detodu OK
[9/209] Diecudey OK
[10/209] Diepami OK
[11/209] Diequipi OK
[12/209] Dithigog OK
[13/209] Dublafer OK
[14/209] Dujomal OK
[15/209] Elagieg OK
[16/209] Golotag OK
[17/209] Grequafie OK
[18/209] Gueshifa OK
[19/209] Kuquelok OK
[20/209] Oduskueb OK
[21/209] Quetode OK
[22/209] Runusath OK
[23/209] Sepigoo OK
[24/209] Stonscuetof OK
[25/209] Suquepog OK
[26/209] Tercippun OK
[27/209] Tiepolem OK
[28/209] Tisupey OK
[29/209] Amifer OK
[30/209] Aruborn OK
[31/209] Asonlig OK
[32/209] Badiswu OK
[33/209] Bibathot OK
[34/209] Bogeebo OK
[35/209] Boudubat OK
[36/209] Burapo OK
[37/209] Butiswu OK
[38/209] Cadedag OK
[39/209] Cefaru OK
[40/209] Cemuquey OK
[41/209] Ceriba OK
[42/209] Ceyebum OK
SKIP Coosimo: no downsampled dir
[44/209] Cornuefor OK
[45/209] Crutaswo OK
[46/209] Dalibul OK
[47/209] Dapafem OK
[48/209] Datokif OK
[49/209] Des